In [1]:
import netCDF4 as nc
import geopandas as gpd
import pandas as pd
import numpy as np
import os
from rasterstats import zonal_stats
from rasterio.transform import from_origin

c:\Users\Tanishq op\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\Tanishq op\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


In [2]:
# 1. Setup
districts = gpd.read_file("india_districts_merged.geojson")
years = range(2014, 2026) # 2014 to 2025
output_data = []

# IMD 0.25-degree grid spatial parameters
# Starts at 6.5N, 66.5E; Last point is 38.5N, 100.0E
# We use a negative lat_step (-0.25) because we start from North and go South
transform = from_origin(66.5, 38.5, 0.25, 0.25)

print("Starting extraction...")

Starting extraction...


In [5]:
# 2. Loop through each year
for year in years:
    filename = f"RF25_ind{year}_rfp25.nc"
    
    if not os.path.exists(filename):
        print(f"Skipping {filename}: File not found.")
        continue
    
    # Open dataset and extract variable
    # IMD NetCDF variable is usually 'RAINFALL'
    with nc.Dataset(filename) as ds:
        # Get the variable name (handles 'RAINFALL', 'rain', etc.)
        # Pick the first 3D variable (time, lat, lon) to avoid selecting TIME by mistake
        var_name = [
            v for v in ds.variables
            if v not in ['lat', 'lon', 'time', 'LONGITUDE', 'LATITUDE', 'TIME']
            and len(ds.variables[v].dimensions) == 3
        ][0]
        data = ds.variables[var_name][:] # Shape: (days, lat, lon)
        
        # Calculate annual total rainfall
        annual_sum = np.sum(data, axis=0)
        
        # 3. Zonal Statistics for all districts
        stats = zonal_stats(
            districts, 
            annual_sum, 
            affine=transform, 
            stats="mean", 
            nodata=-999.0 # Typical IMD missing value
        )
        
        # 4. Store results
        for i, s in enumerate(stats):
            output_data.append({
                "District": districts.iloc[i]['NAME_2'], # Verify this key in your geojson
                "Year": year,
                "Annual_Rainfall_mm": s['mean']
            })
    
    print(f"Processed Year: {year}")
    
df = pd.DataFrame(output_data)
df.to_csv("district_rainfall_2014_2025.csv", index=False)
print("Extraction completed. Data saved to district_rainfall_2014_2025.csv")

Processed Year: 2014
Processed Year: 2015
Processed Year: 2016
Processed Year: 2017
Processed Year: 2018
Processed Year: 2019
Processed Year: 2020
Processed Year: 2021
Processed Year: 2022
Processed Year: 2023
Processed Year: 2024
Processed Year: 2025
Extraction completed. Data saved to district_rainfall_2014_2025.csv
